In [5]:
import sys
import logging
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

PROJECT_ROOT = Path.cwd().parent
SRC_PATH     = PROJECT_ROOT / "src"
DATA_PATH    = PROJECT_ROOT / "data" / "processed" / "train.parquet"
MODEL_PATH   = PROJECT_ROOT / "models" / "pyspark"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from trainers import pyspark_trainer as sparkt

In [4]:
spark = sparkt.start_spark_session(
    app_name="Avazu-CTR-MLP",
    memory_gb=10,
    cores=8,
)

train_df = spark.read.parquet(str(DATA_PATH))
test_df  = spark.read.parquet(str(TEST_PATH))

print(f"train_df: {train_df.count():,} rows")
print(f"test_df:  {test_df.count():,} rows")

In [ ]:
required = set(
    DICHOTOMIZE
    + list(TOP_K.keys())
    + TE_HYBRID
    + TE_PURE
    + NUMERIC_SCALED
    + NUMERIC_PASSTHROUGH
    + [TIME_BAND, TARGET]
)
present = set(train_df.columns)

missing = required - present
if missing:
    print(f"Faltan columnas: {missing}")
else:
    print("Todas las columnas requeridas están presentes")

In [ ]:
param_grid = {
    "layers":   [(50,), (100,), (100, 50)],
    "stepSize": [0.01, 0.001],
    "maxIter":  [100, 200, 500],
}

df_results = sparkt.run_grid_search_spark(
    train_df=train_df,
    param_grid=param_grid,
    model_path=MODEL_PATH,
    results_filename="grid_spark_s10.json",
    smoothing=10,
    num_folds=3,
    seed=42,
)

# Prueba

In [ ]:
test_grid = {
    "layers":   [(50,)],
    "stepSize": [0.001],
    "maxIter":  [100],
}

df_test = sparkt.run_grid_search_spark(
    train_df=train_df,
    param_grid=test_grid,
    model_path=MODEL_PATH,
    results_filename="grid_spark_test.json",
    smoothing=10.0,
    num_folds=2,
)

print(df_test[["mean_auc", "std_auc", "mean_fit_time"]])